In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

In [ ]:
from glob import glob
from os.path import join as opj
import pandas as pd
import gzip

In [ ]:
def load_data(fn, perturb_width):
    with gzip.open(fn) as fd:
        data = torch.load(fd)
    return pd.DataFrame({"path": data["path"], "embeddings":  data["embeddings"].tolist(), "perturb_width": perturb_width})

In [ ]:
def batched_heatmap_to_rgb(heatmaps: np.ndarray) -> np.ndarray:
    """
    Convert batched heatmaps (values in [0,1]) to RGB using viridis colormap.
    
    Args:
        heatmaps: np.ndarray of shape (B, L, A, H, W)

    Returns:
        np.ndarray of shape (B, L, A, H, W, 3), dtype uint8
    """
    cmap = plt.get_cmap('viridis')
    B, L, A, H, W = heatmaps.shape
    flat = heatmaps.reshape(-1, H, W)
    rgba = cmap(flat)               # (B*L*A, H, W, 4)
    rgb = (rgba[..., :3] * 255).astype(np.uint8)
    return rgb.reshape(B, L, A, H, W, 3)

In [ ]:
#preds = pd.DataFrame(glob("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/*BENCHDB_BS*/predictions/val_predictions.pt.gz"), columns=["pred_fn"])
preds = pd.DataFrame(glob("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/*BENCHDB_BS*/predictions/val_predictions.pt.gz"), columns=["pred_fn"])

#preds = pd.DataFrame(glob("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/77069427_May26-10-42-49_sd1000_dev_tune0/models/eval/training_124999/*BENCHDB_BS*/predictions/val_predictions.pt.gz"), columns=["pred_fn"])



preds

In [ ]:
preds["band_width"] = preds["pred_fn"].apply(lambda x:int(x.split("/")[-3].split("_")[5].removeprefix("BS")))
preds = preds.sort_values("band_width")
pred_data = [load_data(i["pred_fn"], i["band_width"]) for _,i in preds.iterrows()]

In [ ]:
all_pred_data = pd.concat([i.iloc[:256] for i in pred_data])
all_pred_data_sorted = all_pred_data.sort_values("path", kind="stable").reset_index(drop=True)
embs = torch.nn.functional.normalize(torch.tensor(all_pred_data_sorted["embeddings"]), dim=1)

In [ ]:
def heatmap_to_rgb(heatmap: np.ndarray, cmap_name='viridis') -> np.ndarray:
    cmap = plt.get_cmap(cmap_name)
    rgba_img = cmap(heatmap)  # shape: (H, W, 4), values in [0, 1]
    rgb_img = (rgba_img[..., :3] * 255).astype(np.uint8)  # drop alpha, convert to uint8
    return rgb_img

In [ ]:
sim3 = embs@embs.T
Image.fromarray(heatmap_to_rgb(sim3)).save("89d3ad98_perturb_all.png")
#plt.colorbar()

In [ ]:
mean3 = torch.mean(torch.stack([sim3[i*7:(i+1)*7, i*7:(i+1)*7] for i in range(256)]), dim=0)
Image.fromarray(heatmap_to_rgb(mean3)).save("89d3ad98_perturb_mean.png")
plt.imshow(mean3)
plt.clim(0, 1)
plt.colorbar()

# Instance discriminatioin

In [ ]:
def get_acc(clean_df, query_df):
    clean = torch.nn.functional.normalize(torch.tensor(clean_df["embeddings"]), dim=1)
    query = torch.nn.functional.normalize(torch.tensor(query_df["embeddings"]), dim=1)
    acc = ((query@clean.T).argmax(dim=1) == torch.arange(len(clean))).sum() / len(clean)
    return acc.item()

In [ ]:
[get_acc(pred_data[0], pred_data[i]) for i in range(7)]

In [ ]:
df = pd.DataFrame({"DINOv2": [0.9906283617019653,
 0.8060129880905151,
 0.1594528704881668,
 0.013082340359687805,
 0.0045774648897349834,
 0.004171181004494429,
 0.004171181004494429],
      "Silica": [0.9906283617019653,
 0.9365384578704834,
 0.7224810123443604,
 0.33854278922080994,
 0.04534127935767174,
 0.0042253523133695126,
 0.004171181004494429], "perturb_width":np.arange(7)})

In [ ]:
df_melted = pd.melt(df, id_vars=['perturb_width'], value_vars=['DINOv2', "Silica"], var_name='method', value_name='acc')


In [ ]:
import altair as alt

In [ ]:
xaxis = alt.Axis(tickSize=0,
                          values=np.arange(7),
                          title="Perturbation width")
yaxis = alt.Axis(tickSize=0,
                 values=np.linspace(0,1,6),
                          title="Instance discrimination accuracy")
alt.Chart(df_melted).mark_line(point=True).encode(
    x=alt.X("perturb_width", axis=xaxis, scale=alt.Scale(domain=[0,6])),
    y=alt.Y("acc:Q",axis=yaxis),
    color=alt.Color('method', legend=alt.Legend(title="Method"))
).properties(width=400,height=400).configure_axis(
    labelFontSize=14,
    titleFontSize=14,
).configure_legend(titleFontSize=14,labelFontSize=14).interactive()#.save("perturb_instance_discrimination.json")


# Get image sample

In [ ]:
from torchvision.transforms.functional import adjust_contrast, adjust_brightness
import einops
from PIL import Image

In [ ]:
ims = torch.stack([adjust_brightness(adjust_contrast(
    torch.load(f"../train/band_perturb{i}.pt")["alt_fg"],
    2), 3) for i in range(7)])

In [ ]:
ims_reshaped = einops.rearrange(ims, "p b c h w -> b p h w c")

In [ ]:
torch.save(ims_reshaped, "perturb_im_viz.pt")

In [ ]:
mkdir perturb_im_viz

In [ ]:
for ii, i in enumerate(ims_reshaped):
    for ij, j in enumerate(i):
        
        Image.fromarray((j*255).numpy().astype(np.uint8)).save(f"perturb_viz_{ii}_{ij}.png")

In [ ]:
for i in range(256):
    plt.figure()
    plt.imshow(einops.rearrange(ims_reshaped[i], "p h w c -> h (p w) c"))